# Submissão B — DeBERTa-v3-base Fine-tuned

Classificação do `subm1.csv` usando o modelo fine-tuned guardado localmente

**Ficheiros necessários na mesma pasta:**
- `deberta_base_finetuned_base/` — pasta com `config.json`, `model.safetensors`, `tokenizer.json`, `tokenizer_config.json`

In [1]:
# -- Imports & Configuração ----------------------------------------
import re
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# --- Ficheiros ----------------------------------------------------------------
PREDICT_CSV = 'subm2.csv'
OUTPUT_CSV  = 'subm2-g5-MECD-A.csv'
MODEL_DIR   = 'BrunoFilipeRDS/deberta_base_finetuned_5_classes'

# --- Hiperparâmetros de inferência -------------------------------------------
MAX_LEN    = 128
BATCH_SIZE = 8

# --- Mapeamento de classes ---------------------
CLASSES   = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
IDX2LABEL = {i: c for i, c in enumerate(CLASSES)}

print('Config OK.')

Device: cuda
Config OK.


In [2]:
# -- Função de limpeza de texto ------------------------------------
def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

print('clean_text definida.')

clean_text definida.


In [3]:
# -- Carregar dataset -----------------------------------------------
df_pred = pd.read_csv(PREDICT_CSV, sep=';', encoding='utf-8-sig')
df_pred.columns = df_pred.columns.str.strip()
df_pred['Text'] = df_pred['Text'].astype(str).apply(clean_text)

print(f'Registos a prever : {len(df_pred)}')
print(df_pred.head(3).to_string(index=False))

Registos a prever : 150
    ID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   Text
D2-101                                                                       Microbial mats of coexisting bacteria and archaea were the dominant form of life in t

In [4]:
# -- Carregar tokenizer + modelo fine-tuned ------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

print(f'Modelo carregado de "{MODEL_DIR}".')
print(f'Número de classes no modelo : {model.config.num_labels}')

config.json: 0.00B [00:00, ?B/s]

c:\Users\bruno\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bruno\.cache\huggingface\hub\models--BrunoFilipeRDS--deberta_base_finetuned_5_classes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Modelo carregado de "BrunoFilipeRDS/deberta_base_finetuned_5_classes".
Número de classes no modelo : 5


In [5]:
# -- Inferência em batches -----------------------------------------
texts      = df_pred['Text'].tolist()
all_preds  = []
all_probs  = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    enc   = tokenizer(
        batch,
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
        return_tensors='pt',
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**enc).logits 
        probs  = torch.softmax(logits, dim=-1)
        preds  = logits.argmax(-1)

    all_preds.extend(preds.cpu().tolist())
    all_probs.extend(probs.cpu().tolist())

    if (i // BATCH_SIZE) % 5 == 0:
        print(f'  Processados {min(i + BATCH_SIZE, len(texts))}/{len(texts)} registos...')

print('Inferência concluída.')

  Processados 8/150 registos...
  Processados 48/150 registos...
  Processados 88/150 registos...
  Processados 128/150 registos...
Inferência concluída.


In [ ]:
# -- Guardar CSV de submissão --------------------------------------
labels_pred = [IDX2LABEL[i] for i in all_preds]

df_out = pd.DataFrame({'ID': df_pred['ID'], 'Labels': labels_pred})
df_out.to_csv(OUTPUT_CSV, index=False, sep=';')

print(f'Ficheiro guardado : {OUTPUT_CSV}')
print()
print('Distribuição de labels:')
print(df_out['Labels'].value_counts().to_string())
print()
print('Primeiras 10 previsões:')
print(df_out.head(10).to_string(index=False))

Ficheiro guardado : subm2-g5-MECD-A.csv

Distribuição de labels:
Labels
OpenAI       71
Google       30
Human        20
Meta         18
Anthropic    11

Primeiras 10 previsões:
    ID Labels
D2-101 OpenAI
D2-102 OpenAI
D2-103 OpenAI
D2-104 OpenAI
D2-105 Google
D2-106 Google
D2-107   Meta
D2-108 OpenAI
D2-109 OpenAI
D2-110 OpenAI


: 